# Week 38, Day 2: Primary visual cortex: oriented receptive fields

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w038/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Orientation tuning of a Gabor cell

Write gabor(theta, f, phase, sx, sy), returning the receptive field on the grid (X, Y in degrees) with orientation theta in degrees. A drifting grating response is computed for you as the mean over eight phases of the rectified dot product.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
n = 64; xy = (np.arange(n) - n / 2) * (4 / n)          # 4 degrees across
X, Y = np.meshgrid(xy, xy)

def gabor(theta, f, phase=0.0, sx=0.4, sy=0.6):
    t = np.deg2rad(theta)
    xr = ...
    yr = ...
    return np.exp(-xr ** 2 / (2 * sx ** 2) - yr ** 2 / (2 * sy ** 2)) * np.cos(2 * np.pi * f * xr + phase)

def grating(theta, f, phase):
    t = np.deg2rad(theta)
    return np.cos(2 * np.pi * f * (X * np.cos(t) + Y * np.sin(t)) + phase)

def drift_response(rf, theta, f):
    return np.mean([max(0.0, np.sum(rf * grating(theta, f, p))) for p in np.linspace(0, 2 * np.pi, 8, endpoint=False)])

rf = gabor(30, 1.0)
oris = np.arange(0, 180, 10)
tuning = np.array([drift_response(rf, o, 1.0) for o in oris])
print("preferred orientation:", oris[np.argmax(tuning)], "deg")
fig, ax = plt.subplots(1, 2, figsize=(8, 3.5))
ax[0].imshow(rf, cmap="RdBu_r", extent=[-2, 2, -2, 2]); ax[0].set_title("receptive field")
ax[1].plot(oris, tuning, "o-"); ax[1].set_xlabel("grating orientation (deg)"); plt.show()

In [ ]:
import numpy as np
def test_peak():
    "the cell prefers 30 degrees"
    assert oris[np.argmax(tuning)] == 30

def test_selective():
    "the response at the orthogonal orientation is under 5% of the peak"
    assert tuning[list(oris).index(120)] < 0.05 * tuning.max()

test_peak()
test_selective()
print("All checks passed")

### Exercise: Spatial frequency tuning and bandwidth

Write bandwidth_octaves(freqs, resp): the width of the response curve at half its maximum, measured as log2 of the ratio of the highest to the lowest frequency where the response is at least half the peak. Compare a Gabor with sx = 0.4 with one with sx = 0.8.

In [ ]:
freqs = np.logspace(np.log10(0.25), np.log10(4), 25)
sf_tuning = np.array([drift_response(gabor(30, 1.0), 30, f) for f in freqs])

def bandwidth_octaves(freqs, resp):
    above = ...
    return ...

bw_narrow = bandwidth_octaves(freqs, sf_tuning)
bw_wide = bandwidth_octaves(freqs, np.array([drift_response(gabor(30, 1.0, sx=0.8), 30, f) for f in freqs]))
print(f"preferred frequency {freqs[np.argmax(sf_tuning)]:.2f} cycles/deg; bandwidth {bw_narrow:.2f} octaves (sx 0.4) and {bw_wide:.2f} octaves (sx 0.8)")
plt.semilogx(freqs, sf_tuning, "o-"); plt.xlabel("spatial frequency (cycles/deg)"); plt.show()

In [ ]:
import numpy as np
def test_peak():
    "the cell prefers about 1 cycle per degree"
    assert 0.8 <= freqs[np.argmax(sf_tuning)] <= 1.25

def test_bandwidth():
    "the wider envelope gives a narrower bandwidth (in octaves)"
    assert bw_wide < bw_narrow

test_peak()
test_bandwidth()
print("All checks passed")

### Exercise: Building a simple cell from the LGN

Build rf_hw as the sum of five ON-centre difference-of-Gaussians fields centred at x = 0 and y = -0.8, -0.4, 0, 0.4, 0.8 degrees (centre σ 0.15, surround σ 0.45, weight 0.9), then compute its drifting-grating tuning at 1.2 cycles per degree and its orientation selectivity index.

In [ ]:
def dog_at(x0, y0, sc=0.15, ss=0.45, w=0.9):
    r2 = (X - x0) ** 2 + (Y - y0) ** 2
    c = np.exp(-r2 / (2 * sc ** 2)); s = np.exp(-r2 / (2 * ss ** 2))
    return c / c.sum() - w * s / s.sum()

rf_hw = ...
hw_oris = np.arange(0, 180, 10)
hw_tuning = np.array([drift_response(rf_hw, o, 1.2) for o in hw_oris])
v = np.sum(hw_tuning * np.exp(2j * np.deg2rad(hw_oris)))
osi = np.abs(v) / hw_tuning.sum()
print(f"preferred orientation {hw_oris[np.argmax(hw_tuning)]} deg (0 = vertical stripes, which vary along x), OSI {osi:.2f}")
fig, ax = plt.subplots(1, 2, figsize=(8, 3.5))
ax[0].imshow(rf_hw, cmap="RdBu_r", extent=[-2, 2, -2, 2]); ax[1].plot(hw_oris, hw_tuning, "o-"); plt.show()

In [ ]:
import numpy as np
def test_vertical():
    "the cell built from a vertical row prefers vertical stripes (0 degrees in this convention)"
    assert hw_oris[np.argmax(hw_tuning)] == 0

def test_selective():
    "its orientation selectivity index is above 0.3"
    assert osi > 0.3

test_vertical()
test_selective()
print("All checks passed")